# Phase 3 — EV Bus Video Analytics

**Project:** Multimodal AI Strategy for EV-Bus Market Disruption in India  
**Notebook:** `05_video_analytics.ipynb`  
**Author:** MBA Group Project — SPJIMR  

## Purpose

This notebook runs a full video analytics pipeline on four EV-bus brand videos:
- **EKA** — EKA Mobility (new-age OEM)
- **OLECTRA** — Olectra Greentech (new-age OEM)
- **SWITCH** — Switch Mobility (new-age OEM)
- **TATA** — Tata Motors (legacy OEM)

## Pipeline Summary

| Stage | Method | Output |
|-------|--------|--------|
| Frame extraction | OpenCV at 1 fps | JPEG keyframes |
| Low-level features | OpenCV (brightness, contrast, saturation, sharpness, edge density, colourfulness, visual balance) | per-frame metrics |
| Shot boundary detection | Pixel-difference between consecutive frames | shot cut timestamps |
| OCR | Tesseract — extract on-screen text | subtitles, lower-thirds, text overlays |
| Sentiment | VADER on OCR text | narrative framing score |
| Object detection | DETR ResNet-50 (if torch available) | bus / person / vehicle counts |
| Scene captioning | BLIP (if torch available) | natural-language frame description |
| Zero-shot classification | CLIP (if torch available) | EV-bus theme scores |
| Aggregation | per-video summaries | brand comparison |
| Charts | matplotlib | timing, visual, theme, sentiment |

## Methodology Notes

- All findings from this pipeline are **exploratory** — small sample (4 videos, 1 per brand).
- Video sentiment is based on **OCR of on-screen text** (subtitles / captions / text overlays), not dialogue.
- CLIP/DETR/BLIP features require `torch` + `transformers`. Without them, the pipeline still runs using OpenCV + OCR.
- Shot-cut detection uses a 90th-percentile pixel-difference threshold — appropriate for varied production styles but may over- or under-count in high-motion or low-motion content.
- OEM brand classifications: EKA and SWITCH = new-age OEMs; OLECTRA = new-age OEM; TATA = legacy OEM.
- Do not interpret video content quality as evidence of bus manufacturing quality.

## Limitations

- 4 videos is too small for statistical inference.
- Brand choice of video to publish is a marketing decision, not an operational one.
- OCR accuracy depends on video subtitle quality and font rendering.
- Findings should be labelled **exploratory** in the final report.


## 0. Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Resolve project paths relative to this notebook's location
# Works whether the notebook is opened from inside notebooks/ or from root
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
VIDEO_ANALYTICS_ROOT = NOTEBOOK_DIR.parent  # Video_Analytics/
PHASE3_ROOT = VIDEO_ANALYTICS_ROOT.parent   # Phase_3_Video_Analytics/

VIDEOS_DIR   = PHASE3_ROOT / "Videos"
SCRIPTS_DIR  = VIDEO_ANALYTICS_ROOT / "scripts"
OUTPUTS_DIR  = VIDEO_ANALYTICS_ROOT / "outputs"
FRAMES_DIR   = OUTPUTS_DIR / "frames"
PERFRAME_DIR = OUTPUTS_DIR / "per_frame"
SUMMARIES_DIR = OUTPUTS_DIR / "summaries"
CHARTS_DIR   = OUTPUTS_DIR / "charts"

# Add scripts dir to path so we can import the library
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Create output directories
for d in [FRAMES_DIR, PERFRAME_DIR, SUMMARIES_DIR, CHARTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Videos directory   : {VIDEOS_DIR}")
print(f"Outputs directory  : {OUTPUTS_DIR}")
print(f"Videos found       : {[p.name for p in sorted(VIDEOS_DIR.glob('*.mp4'))]}")

In [ ]:
# Install any missing packages (run this cell if you get ImportError below)
# Uncomment and run if needed:
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install",
#                 "opencv-python", "Pillow", "pandas", "matplotlib",
#                 "vaderSentiment", "scikit-learn", "scipy",
#                 "pytesseract", "wordcloud", "nltk"], check=True)
# # Optional — enables BLIP / DETR / CLIP (requires ~2GB download):
# subprocess.run([sys.executable, "-m", "pip", "install",
#                 "torch", "torchvision", "transformers"], check=True)
pass

In [ ]:
from video_analytics_lib import (
    _TORCH_AVAILABLE,
    get_video_metadata,
    process_video,
    summarise_video,
    generate_all_charts,
    _EV_BUS_THEME_KEYWORDS,
)
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print(f"Library loaded.")
print(f"Deep models (BLIP / DETR / CLIP) : {'ENABLED' if _TORCH_AVAILABLE else 'DISABLED — install torch + transformers to enable'}")

## 1. Video Metadata

In [ ]:
video_files = sorted(VIDEOS_DIR.glob("*.mp4"))
print(f"Found {len(video_files)} video files:\n")

meta_rows = []
for vf in video_files:
    m = get_video_metadata(vf)
    meta_rows.append(m)
    print(f"  {m['brand']:<10}  {m['duration_seconds']:>6.1f}s  "
          f"{m['resolution']}  {m['fps']} fps  {m['file_size_mb']} MB")

meta_df = pd.DataFrame(meta_rows)
meta_df

## 2. Frame Extraction + Per-Frame Analysis

We sample **1 frame per second** (configurable). For a 60-second video this gives ~60 keyframes. Each frame goes through:

- Low-level visual features (OpenCV)
- Shot-cut detection via inter-frame pixel difference
- OCR text extraction (Tesseract)
- VADER sentiment on OCR text
- *(if torch available)*: DETR object detection, BLIP caption, CLIP zero-shot

> ⚠️ **Runtime:** With 4 videos at 1 fps and deep models OFF, this takes ~1–2 minutes.  
> With deep models ON, expect 5–15 minutes depending on GPU availability.

In [ ]:
SAMPLE_FPS = 1.0   # frames per second to sample
MAX_FRAMES = 200   # hard cap per video

all_frames_list = []
per_video_dfs = {}

for vf in video_files:
    brand = vf.stem.upper()
    frames_dir = FRAMES_DIR / brand.lower()
    
    df = process_video(
        vf, frames_dir,
        sample_fps=SAMPLE_FPS,
        max_frames=MAX_FRAMES,
        verbose=True,
    )
    
    # Save per-frame CSV
    csv_path = PERFRAME_DIR / f"{brand.lower()}_frames.csv"
    df.to_csv(csv_path, index=False)
    
    per_video_dfs[brand] = df
    all_frames_list.append(df)
    print(f"  ✓ {brand}: {len(df)} frames → {csv_path.name}")

# Combine into master dataframe
master_df = pd.concat(all_frames_list, ignore_index=True)
master_csv = OUTPUTS_DIR / "master_video_analysis.csv"
master_df.to_csv(master_csv, index=False)
print(f"\nMaster CSV: {master_csv.name}  ({len(master_df)} rows × {len(master_df.columns)} cols)")

## 3. Per-Video Aggregation

In [ ]:
summaries = []
for brand, df in per_video_dfs.items():
    s = summarise_video(df)
    summaries.append(s)
    # Save JSON
    jp = SUMMARIES_DIR / f"{brand.lower()}_summary.json"
    with open(jp, "w") as f:
        json.dump(s, f, indent=2, default=str)

print("Brand summaries:")
compare = pd.DataFrame([
    {
        "Brand": s["brand"],
        "Duration (s)": s.get("duration_seconds"),
        "Frames": s.get("total_frames_analysed"),
        "Shot Cuts": s.get("shot_cuts"),
        "Cuts/min": s.get("cuts_per_minute"),
        "Avg Brightness": s.get("avg_brightness"),
        "Avg Saturation": s.get("avg_saturation"),
        "Avg Sharpness": s.get("avg_sharpness"),
        "OCR Text %": s.get("ocr_text_pct"),
        "Avg VADER": s.get("avg_vader_compound"),
        "Dominant Theme": s.get("dominant_text_theme"),
    }
    for s in summaries
])
compare.set_index("Brand", inplace=True)
compare

## 4. Charts

All charts are also saved to `outputs/charts/` as PNG files.

In [ ]:
chart_paths = generate_all_charts(
    master_df, summaries, CHARTS_DIR, per_video_dfs=per_video_dfs
)
print(f"Generated {len(chart_paths)} charts:")
for p in chart_paths:
    print(f"  {p.name}")

In [ ]:
# Display charts inline
from IPython.display import display, Image as IPImage

# Shot pacing
display(IPImage(filename=str(CHARTS_DIR / "shot_pacing.png"), width=700))

In [ ]:
# Visual metrics radar
display(IPImage(filename=str(CHARTS_DIR / "visual_metrics_radar.png"), width=600))

In [ ]:
# Text themes heatmap
display(IPImage(filename=str(CHARTS_DIR / "text_themes_heatmap.png"), width=900))

In [ ]:
# Sentiment distribution
display(IPImage(filename=str(CHARTS_DIR / "sentiment_comparison.png"), width=700))

In [ ]:
# Visual distribution boxplots
display(IPImage(filename=str(CHARTS_DIR / "visual_distribution_boxplots.png"), width=900))

In [ ]:
# Brightness timelines (per brand)
for brand in per_video_dfs:
    p = CHARTS_DIR / f"brightness_timeline_{brand.lower()}.png"
    if p.exists():
        print(f"\n{brand}")
        display(IPImage(filename=str(p), width=900))

In [ ]:
# CLIP perception radar (only if torch was available)
p = CHARTS_DIR / "perception_radar.png"
if p.exists():
    display(IPImage(filename=str(p), width=650))

In [ ]:
# Word clouds
for brand in per_video_dfs:
    p = CHARTS_DIR / f"wordcloud_{brand.lower()}.png"
    if p.exists():
        print(f"\n{brand} — On-Screen Text Word Cloud")
        display(IPImage(filename=str(p), width=750))

## 5. OCR Text Deep-Dive — Top Keywords per Brand

In [ ]:
import re
from collections import Counter

STOP_WORDS = {
    "the", "and", "for", "with", "from", "you", "are", "this", "that",
    "they", "their", "has", "have", "been", "not", "but", "all", "any",
    "was", "were", "its", "can", "will", "more", "than", "into", "who",
    "how", "what", "why", "when", "our", "your", "new", "now", "get",
    "www", "http", "https", "com", "also", "just", "one", "two",
}

for brand, df in per_video_dfs.items():
    text = " ".join(df["ocr_text"].fillna("").tolist()).lower()
    tokens = re.sub(r"[^a-z\s]", " ", text).split()
    tokens = [w for w in tokens if len(w) > 2 and w not in STOP_WORDS]
    top20 = Counter(tokens).most_common(20)
    print(f"\n{brand} — Top 20 OCR words:")
    for word, cnt in top20:
        print(f"  {word:<20} {cnt}")

## 6. Shot Boundary Analysis

In [ ]:
fig, axes = plt.subplots(len(per_video_dfs), 1,
                          figsize=(14, 3 * len(per_video_dfs)), sharex=False)
if len(per_video_dfs) == 1:
    axes = [axes]

BRAND_COLORS = {"EKA": "#1a73e8", "OLECTRA": "#34a853",
                "SWITCH": "#fbbc04", "TATA": "#ea4335"}

for ax, (brand, df) in zip(axes, per_video_dfs.items()):
    color = BRAND_COLORS.get(brand, "#888")
    ax.plot(df["timestamp_s"], df["pixel_diff"], color=color, linewidth=1, alpha=0.7)
    cuts = df[df["is_shot_cut"]]
    ax.vlines(cuts["timestamp_s"], 0, df["pixel_diff"].max(),
              colors="crimson", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_title(f"{brand} — Inter-frame Pixel Difference (shot cuts in red)")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Mean pixel diff")
    ax.grid(alpha=0.2)

plt.tight_layout()
save_path = CHARTS_DIR / "shot_boundary_detail.png"
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path.name}")

## 7. Text-Theme Heatmap (Detail)

In [ ]:
theme_data = {s["brand"]: s.get("text_themes", {}) for s in summaries}
theme_df = pd.DataFrame(theme_data).T.fillna(0).astype(int)
theme_df.columns = [c.replace("_", "\n") for c in theme_df.columns]

print("\nTheme keyword counts per brand:\n")
print(theme_df.to_string())

## 8. Strategic Interpretation

> ⚠️ **Limitations reminder:** 4 videos, exploratory only. Interpret as communication-style signals, not evidence of product quality or operational capability.

Below we print a structured interpretation following the project's evidence template:

```
Finding → Metric → Interpretation → Business Implication → Recommended Action
```

In [ ]:
# Automated strategic summary per brand
for s in summaries:
    brand = s["brand"]
    print("=" * 60)
    print(f"  {brand}")
    print("=" * 60)
    print(f"  Duration             : {s.get('duration_seconds', 'N/A')}s")
    print(f"  Editing pace         : {s.get('cuts_per_minute', 'N/A')} cuts/min")
    print(f"  Avg brightness       : {s.get('avg_brightness', 'N/A')}")
    print(f"  Avg colourfulness    : {s.get('avg_colourfulness', 'N/A')}")
    print(f"  OCR text frames      : {s.get('ocr_text_pct', 'N/A')}%")
    print(f"  Dominant OCR theme   : {s.get('dominant_text_theme', 'N/A')}")
    print(f"  Avg sentiment (VADER): {s.get('avg_vader_compound', 'N/A')}")
    
    perc = s.get("perception_scores", {})
    if perc:
        print(f"  CLIP perception:")
        for k, v in perc.items():
            print(f"    {k.replace('pxy_',''):<22}: {v:.1f}/100")
    
    theme_hits = s.get("text_themes", {})
    top_themes = sorted(theme_hits.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"  Top text themes      : {', '.join(f'{t}({c})' for t, c in top_themes)}")
    print()

## 9. Save Final Run Summary JSON

In [ ]:
from datetime import datetime

run_summary = {
    "notebook": "05_video_analytics.ipynb",
    "run_timestamp": datetime.now().isoformat(),
    "videos_dir": str(VIDEOS_DIR),
    "outputs_dir": str(OUTPUTS_DIR),
    "sample_fps": SAMPLE_FPS,
    "max_frames_per_video": MAX_FRAMES,
    "deep_models_used": _TORCH_AVAILABLE,
    "brands_processed": [s["brand"] for s in summaries],
    "total_frames_analysed": int(master_df.shape[0]),
    "charts_generated": [p.name for p in chart_paths],
    "summaries": summaries,
    "limitations": [
        "Exploratory only — 4 brand videos is not a statistically valid sample.",
        "Video content quality ≠ bus manufacturing quality.",
        "OCR sentiment reflects on-screen text framing, not passenger or operator experience.",
        "Shot-cut detection uses a heuristic threshold; results may vary by video style.",
        "CLIP/BLIP/DETR features not available without torch + transformers.",
    ]
}

summary_path = OUTPUTS_DIR / "pipeline_run_summary.json"
with open(summary_path, "w") as f:
    json.dump(run_summary, f, indent=2, default=str)

print(f"Run summary saved: {summary_path}")

# Also save brand comparison CSV
compare_rows = []
for s in summaries:
    row = {k: v for k, v in s.items() if not isinstance(v, (dict, list))}
    for theme, cnt in s.get("text_themes", {}).items():
        row[f"theme_{theme}"] = cnt
    compare_rows.append(row)
compare_path = SUMMARIES_DIR / "brand_comparison.csv"
pd.DataFrame(compare_rows).to_csv(compare_path, index=False)
print(f"Brand comparison CSV saved: {compare_path}")

## 10. Output File Index

In [ ]:
import os

print("=" * 60)
print("  OUTPUT FILES")
print("=" * 60)
for root, dirs, files in os.walk(OUTPUTS_DIR):
    # Skip frame JPEG directories (too many files)
    if "frames" in root and any(f.endswith(".jpg") for f in files):
        rel = Path(root).relative_to(OUTPUTS_DIR)
        print(f"  {rel}/   [{len(files)} JPEG frames]")
        continue
    for fname in sorted(files):
        fpath = Path(root) / fname
        rel = fpath.relative_to(OUTPUTS_DIR)
        size_kb = fpath.stat().st_size // 1024
        print(f"  {str(rel):<55} {size_kb:>6} KB")

print("\nDone. All outputs written to Video_Analytics/outputs/")